# 11 - Math For Robotics

## What / Why / How

**What we are trying to do:** Review the math tools that repeatedly appear in robotics: projections, least squares, covariance, and Gaussian uncertainty.

**Why this matters:** Robotics papers and systems use these ideas constantly. Understanding them makes estimation, calibration, control, and learning much less mysterious.

**How we will do it:** Work through small numeric examples, fit a noisy line, compute covariance, and interpret uncertainty as geometry.

## Goal

Robotics uses math as a working language. This notebook reviews the pieces that appear everywhere:

- Vectors and projections
- Least squares
- Covariance and uncertainty ellipses
- Gaussian probability intuition

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
a = np.array([3.0, 1.0])
b = np.array([1.0, 2.0])
projection_of_a_on_b = b * (a @ b) / (b @ b)
residual = a - projection_of_a_on_b
print("projection:", projection_of_a_on_b)
print("residual is orthogonal:", round(float(residual @ b), 8))

In [ ]:
rng = np.random.default_rng(1)
x = np.linspace(-2, 2, 30)
y = 1.7 * x - 0.4 + rng.normal(0, 0.25, size=len(x))
A = np.c_[x, np.ones_like(x)]
slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
print("estimated line: y =", round(float(slope), 3), "* x +", round(float(intercept), 3))

In [ ]:
samples = rng.multivariate_normal(mean=[1, 2], cov=[[0.5, 0.3], [0.3, 0.8]], size=500)
mean = samples.mean(axis=0)
covariance = np.cov(samples.T)
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
print("mean:", mean)
print("covariance:\n", covariance)
print("uncertainty axes:", np.sqrt(eigenvalues))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 5))
    plt.scatter(samples[:, 0], samples[:, 1], s=8, alpha=0.25)
    plt.scatter([mean[0]], [mean[1]], c="tab:red")
    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.title("Gaussian samples as uncertainty")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Replace least squares with a noisy quadratic fit.
2. Change covariance off-diagonal terms and explain correlation.
3. Explain why uncertainty is a matrix, not just one number.